In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
api_key=os.getenv("GOOGLE_API_KEY")
)

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import GeminiRAG


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = GeminiRAG(
    index=index,
    llm_client=client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, follow these steps:

1.  **Install Ollama:**
    *   Visit [https://ollama.com/download](https://ollama.com/download).
    *   **macOS**: Download the `.pkg` and install it.
    *   **Windows**: Download the `.msi` and install it.
    *   **Linux**: Run the following command in your terminal:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```

2.  **Run a model locally:**
    *   Once installed, open a terminal and type:
        ```bash
        ollama run llama3
        ```
        This command will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3.  **Test the Ollama local server (optional):**
    *   Run the command:
        ```bash
        curl http://localhost:11434
        ```
    *   You should receive a JSON response similar to `{"models": [...]}`.

4.  **Install the Python client (optional, for programmatic use):**
    *   ```bash
        pip install ollama
        ```
    *   You can then use 

In [5]:
prompt = "I just discovered the course. Can I join it?"

In [6]:


response = client.models.generate_content(
model="gemini-2.5-flash",
contents=prompt
)

print(response.text)


That's great you found a course you're interested in!

To tell you if you can join, I'll need a bit more information about the course. The possibility of joining usually depends on:

1.  **Which specific course it is:** What is the name of the course?
2.  **Where it's offered:** Is it through a university, an online platform (like Coursera, Udemy, edX), a private institution, a community college, etc.?
3.  **Its start date and enrollment period:** Has the course already started? Is it self-paced with continuous enrollment, or does it have specific start and end dates with enrollment deadlines?
4.  **Enrollment availability:** Is it full? Are there still spots open?
5.  **Prerequisites:** Does it require specific prior knowledge, degrees, or other courses?
6.  **Application process:** Is it direct enrollment, or do you need to apply?

**Once you tell me the name of the course and where you discovered it, I can often help you find out the specific enrollment details!**

In general, here'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
from google.genai import types

search_tool = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters={
        
        "type": "OBJECT",
        "properties": {
            "query": {
                "type": "STRING",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"]
    }
)

In [9]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=[search_tool]
            )
        ]
    )
)

In [10]:
response

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              name=<... Max depth ...>
            ),
            thought_signature=b'\n\x9a\x03\x01\x0c9\xd6\xc7\xb0\x8b\xbf\xf5\xa6\x96\xa4r\xf6h\x15\xe2z\xd02wW\x94\x86C\xab\xa4{\xd9s{\x00\xfb\x7f\xe0)\x01tW\x13BF3=\x17@:\xc0\x14]m\xd9\xc8\xad\xcey\xfe\xcd\x9c\x7f\x11S\xfb\xf1\xd6\xea3\xdd*Ll\x871D\x16W;\xb2r\xec?\xae\xb7\xd4\xeaX\x17\x8e:\\\xb7\x07\x02\xaf...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='g6I5avLsDZPFnsEP0oSDwA4',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=14,
    prompt_token_count=63,
    prompt_tokens_details=[
      ModalityTokenCount(
     

In [11]:
function_calls = response.function_calls

len(function_calls)

1

In [12]:
call = function_calls[0]
call

FunctionCall(
  args={
    'query': 'join course'
  },
  name='search'
)

In [13]:
args = call.args
args

{'query': 'join course'}

In [14]:
results = search(**args)

In [15]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and Git

In [16]:
import json
result_json = json.dumps(results, indent=2)

In [17]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTa

In [18]:
function_response = types.Part.from_function_response(
    name=call.name,
    response={
        "result": result_json
    }
)

In [19]:
prompt

'I just discovered the course. Can I join it?'

In [20]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=[search_tool]
            )
        ],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True
        )
    )
)

In [21]:
call = response.function_calls[0]
args = call.args

In [22]:
result = search(args["query"])

In [23]:
function_response = types.Part.from_function_response(
    name=call.name,
    response={
        "result": result
    }
)


In [24]:
final_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)]
        ),
        response.candidates[0].content,
        types.Content(
            role="tool",
            parts=[function_response]
        )
    ]
)

In [25]:
final_response.text

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [26]:
def make_call(call):
    args = call.args

    if call.name == "search":
        result = search(**args)
    else:
        raise ValueError(f"Unknown function: {call.name}")

    return types.Part.from_function_response(
        name=call.name,
        response={
            "result": result
        }
    )

In [27]:
instructions = """
You're a course teaching assistant.

You're given a question from a course student and your task is to answer it.

If you need additional information, use the search function.

When performing the first search, use as many keywords from the user's question as possible.

You may perform multiple searches.

Expand your search using new keywords based on the results returned by previous searches.

Provide a clear and concise answer based on the retrieved information.

At the end of your response, ask whether there are other areas the user would like to explore.
"""

question = "I just discovered the course. Can I join it?"

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=question,
    config=types.GenerateContentConfig(
        system_instruction=instructions,
        tools=[
            types.Tool(
                function_declarations=[search_tool]
            )
        ]
    )
)



In [28]:
call.name

'search'

In [29]:
# Check whether Gemini requested any tool calls

if response.function_calls:
    for call in response.function_calls:
        print("function_call:", call.name, call.args)

        function_response = make_call(call)

        final_response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[types.Part.from_text(text=question)]
                ),
                response.candidates[0].content,
                types.Content(
                    role="tool",
                    parts=[function_response]
                )
            ]
        )

        print("ASSISTANT:")
        print(final_response.text)

else:
    print("ASSISTANT:")
    print(response.text)


function_call: search {'query': 'join course enrollment'}
ASSISTANT:
Yes, you can still join the course!

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available, and the deadlines are listed in the course management platform.


In [30]:
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(text=question)
        ]
    )
]

config = types.GenerateContentConfig(
    system_instruction=instructions,
    tools=[
        types.Tool(
            function_declarations=[search_tool]
        )
    ],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=contents,
        config=config
    )

    if response.function_calls:
        has_function_calls = True

        # Add Gemini's function-call request to the conversation history
        contents.append(response.candidates[0].content)

        for call in response.function_calls:
            print("function_call:", call.name, call.args)

            function_response = make_call(call)

            # Add tool result back to the conversation history
            contents.append(
                types.Content(
                    role="tool",
                    parts=[function_response]
                )
            )

    else:
        print("ASSISTANT:")
        print(response.text)

    it += 1

    if not has_function_calls:
        break


iteration #1...
function_call: search {'query': 'join course enrollment'}
iteration #2...
ASSISTANT:
Yes, you can still join the course. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted. The course materials are available, and you can start whenever you want. The next offering of the course will be in Summer 2027.

Is there anything else you'd like to explore about the course?


In [31]:
def agent_loop(instructions, question, model="gemini-2.5-flash") -> str:
    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=question)
            ]
        )
    ]

    config = types.GenerateContentConfig(
        system_instruction=instructions,
        tools=[
            types.Tool(
                function_declarations=[search_tool]
            )
        ],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True
        )
    )

    it = 1
    last_answer = ""

    while True:
        print(f"iteration #{it}...")

        response = client.models.generate_content(
            model=model,
            contents=contents,
            config=config
        )

        if response.function_calls:
            contents.append(response.candidates[0].content)

            for call in response.function_calls:
                print("function_call:", call.name, call.args)

                function_response = make_call(call)

                contents.append(
                    types.Content(
                        role="tool",
                        parts=[function_response]
                    )
                )

        else:
            print("ASSISTANT:")
            last_answer = response.text
            print(last_answer)
            break

        it += 1

    return last_answer


In [32]:
# instructions = """
# You're a course teaching assistant.
# You're given a question from a course student and your task is to answer it.

# If you want to look up information, use the search function. 
# Use as many keywords from the user question as possible when making first requests.

# Make multiple searches. First perform search, analyze the results 
# and then perform more searchers. 

# At the end, ask if there are other areas that the user wants to explore.
# """
instructions = """
You're a course teaching assistant.

Use the search function before answering.

You must perform at least two separate searches before giving a final answer.

For the first search, use keywords from the user's question.

For the second search, use new keywords based on the first search results.

Do not answer until at least two search results have been received.

At the end, ask if there are other areas the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [33]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {'query': 'join course'}
iteration #2...
function_call: search {'query': 'project submission deadline'}
iteration #3...
ASSISTANT:
Yes, you can join the course!

If you wish to receive a certificate, you will need to submit your project while submissions are still being accepted. The specific deadlines for project submission are listed on the course management platform. You can find more details, including how to start and follow the weekly workflow, on the LLM Zoomcamp docs, general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

Do you have any other areas you'd like to explore about the course?


In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {'query': 'queen gambit'}
iteration #2...
ASSISTANT:
I can only answer questions about the course or its logistics. Is there anything else I can help you with regarding the course material?


In [35]:
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [36]:
from toyaikit_adaptation.llm import GeminiClient

In [37]:
llm = GeminiClient(model="gemini-2.5-flash")

response = llm.send_request(
    chat_messages=[
        "You are a helpful assistant.",
        "Explain what ToyAIKit does."
    ]
)

print(response.text)

**ToyAIKit** is a conceptual name (or a potential real project name for a specific library) for a **lightweight, simplified toolkit or library designed to introduce individuals to the fundamentals of Artificial Intelligence (AI) and Machine Learning (ML).**

Think of it as **"training wheels" for AI.**

Its primary goal is to **demystify complex AI concepts** and make them **accessible to beginners, students, hobbyists, or developers new to the field** who might find full-fledged frameworks like TensorFlow, PyTorch, or even scikit-learn overwhelming at first.

Here's a breakdown of what a ToyAIKit typically does:

1.  **Simplifies Common AI Tasks:** It provides a user-friendly interface (APIs) for performing core AI operations, reducing the amount of complex code a beginner needs to write.
2.  **Implements Core Algorithms:** You'll typically find implementations of foundational machine learning algorithms such as:
    *   Linear Regression
    *   Logistic Regression
    *   Decision T

In [38]:
from toyaikit_adaptation.tools import GeminiTools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit_adaptation.runners import GeminiRunner
from toyaikit_adaptation.llm import GeminiClient

In [39]:
agent_tools = GeminiTools()
agent_tools.add_tool(search, search_tool)

In [40]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [41]:
agent_tools = GeminiTools()
agent_tools.add_tool(search)

In [42]:
agent_tools.get_tools()

[Tool(
   function_declarations=[
     FunctionDeclaration(
       description='Search the FAQ database for entries matching the given query.',
       name='search',
       parameters=Schema(
         properties={
           'query': Schema(
             description='query parameter',
             type=<Type.STRING: 'STRING'>
           )
         },
         required=[
           'query',
         ],
         type=<Type.OBJECT: 'OBJECT'>
       )
     ),
   ]
 )]

In [43]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [70]:
instructions = """
You're a course teaching assistant.

You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

IMPORTANT: If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests. 
When constructing search queries, correct obvious spelling mistakes and typographical errors, particularly for technical terms. If a term does not correspond to a known technical concept, tool, framework, or keyword, infer and use the most likely intended technical term.
For example, "doker" → "Docker", - "olama" → "Ollama"

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.

"""

In [71]:
llm_client = GeminiClient(model="gemini-2.5-flash")

runner = GeminiRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=IPythonChatInterface(),
    llm_client=llm_client,
)

result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)


-> Response received

=== TOOL CALL ===
name: search
arguments: {"query": "Ollama locally"}

=== RETRIEVED RESULTS ===

[1]
id: 1d0b969028
course: llm-zoomcamp
section: Module 1: RAG
question: Ollama: How to install Ollama?
answer: First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:

- **macOS**: Download the `.pkg` and install it.
- **Windows**: Download the `.msi` and install it.
- **Linux**: Run the following command in the terminal:

  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

Once installed, open a terminal and type:

```bash
ollama run llama3
```

This command will:

- Download the LLaMA 3 model (~4GB).
- Start the model locally.
- Open a chat-like interface where you can type questions.

To test the Ollama local server, run the following command:

```bash
curl http://localhost:11434
```

You should receive a response similar to:

```json
{"models": [...]}  
```

Then, install the Py

-> Response received


In [51]:
print(result.last_message)



Is there anything else you'd like to explore?


In [52]:
result.cost

CostInfo(input_cost=Decimal('0.000489'), output_cost=Decimal('0.000885'), total_cost=Decimal('0.001374'))

In [53]:
result.all_messages

[Content(
   parts=[
     Part(
       text="""System instruction: 
 You're a course teaching assistant.
 You're given a question from a course student and your task is to answer it.
 
 If you want to look up information, use the search function. 
 Use as many keywords from the user question as possible when making first requests.
 
 Make multiple searches. First perform search, analyze the results 
 and then perform more searchers. 
 
 The question has to be about the course or its logistics, offtopic questions 
 shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
 If you can't answer the question using FAQ, don't do it yourself. Only use the 
 facts from the FAQ database.
 
 At the end, ask if there are other areas that the user wants to explore.
 """
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text='How do I run Ollama locally?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=Funct

In [54]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received

=== TOOL CALL ===
name: search
arguments: {"query": "Ollama run different model"}

=== RETRIEVED RESULTS ===

[1]
id: 1d0b969028
course: llm-zoomcamp
section: Module 1: RAG
question: Ollama: How to install Ollama?
answer: First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:

- **macOS**: Download the `.pkg` and install it.
- **Windows**: Download the `.msi` and install it.
- **Linux**: Run the following command in the terminal:

  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

Once installed, open a terminal and type:

```bash
ollama run llama3
```

This command will:

- Download the LLaMA 3 model (~4GB).
- Start the model locally.
- Open a chat-like interface where you can type questions.

To test the Ollama local server, run the following command:

```bash
curl http://localhost:11434
```

You should receive a response similar to:

```json
{"models": [...]}  
```

Then, in

-> Response received


In [56]:
runner.run();

You: docker


-> Response received

=== TOOL CALL ===
name: search
arguments: {"query": "docker"}

=== RETRIEVED RESULTS ===

[1]
id: 66ccbb7da0
course: llm-zoomcamp
section: Module 5: Monitoring
question: How can I remove all Docker containers, images, and volumes, and builds from the terminal?
answer: 1. Delete all containers (including running ones):

```bash
docker rm -f $(docker ps -aq)
```

2. Remove all images:

```bash
docker rmi -f $(docker images -q)
```

3. Delete all volumes:

```bash
docker volume rm $(docker volume ls -q)
``` 

[2]
id: e8df9f0d12
course: llm-zoomcamp
section: Module 6: Best Practices
question: Docker: When trying to run a streamlit app using docker-compose, I get: Error response from daemon: failed to create task for container: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: exec: "streamlit": executable file not found in $PATH: unknown. The app runs fine outside of docker-compose
answer: To resolve this iss

-> Response received


KeyboardInterrupt: Interrupted by user